# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_URL = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

loader = PyPDFLoader(PDF_URL)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Pages loaded: {len(docs)}")
print(document_text[:500])

Pages loaded: 26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
from pydantic import BaseModel, Field

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int = 0
    OutputTokens: int = 0

TONE = "African-American Vernacular English"
MODEL = "gpt-4o-mini"

developer_prompt = f"""
You are an expert document analyst and skilled writer.
Write the Summary field entirely in {TONE}.
Write the Relevance field in plain professional English.
The Tone field must say exactly: "{TONE}".
Do not invent information. All claims must come from the document.
Set InputTokens and OutputTokens to 0.
"""

user_prompt = f"""
Analyze the document below and return a structured summary.

<document>
{document_text}
</document>
"""

response = client.responses.parse(
    model=MODEL,
    instructions=developer_prompt,
    input=user_prompt,
    text_format=SummaryOutput,
)

result = response.output_parsed
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

print(result.model_dump_json(indent=2))

{
  "Author": "MIT NANDA",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This document analyzes the current impact of Generative AI (GenAI) in various industries, highlighting a significant divide where many organizations fail to achieve measurable returns on their AI investments. It explores reasons for this divide and identifies successful strategies for overcoming challenges in AI implementation.",
  "Summary": "Ayo, listen up. So this report's layin' it down 'bout how most companies messin' with GenAI ain't seein' no real change or bread comin' from it. Like, they droppin' mad cash, 30 to 40 billion, but 95% ain't gettin' nothin' back. They callin' it the GenAI Divide 'cause while folks is tryin' tools like ChatGPT, it ain't doin' much for makin' that cash flow. Only a few groups really winnin' by usin' these tools right. The big problem ain't even the tech itself—it's how folks approach usin' it. Too many pilots stallin' out 'cause they ain't pickin'

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel

# Connect DeepEval to the course API gateway

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Define Summarization Metric with five bespoke assessment questions

summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary capture the divide between AI leaders and laggards?",
        "Does the summary mention key barriers organizations face in adopting AI?",
        "Does the summary reflect findings on AI investment levels and expected ROI?",
        "Does the summary address workforce impact or skills gaps related to AI?",
        "Does the summary include the report's forward-looking recommendations?",
    ],
    model=model,
    threshold=0.5,
)

# Define three G-Eval metrics with five evaluation steps each

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate whether the summary is logically structured and easy to follow.",
    evaluation_steps=[
        "Check that the summary has a clear opening, middle, and conclusion.",
        "Verify that ideas flow naturally without abrupt topic changes.",
        "Assess whether the summary contains any contradictions or inconsistencies.",
        "Determine whether the summary stays focused on the central topic.",
        "Check that the level of detail is consistent throughout.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.5,
)

tonality_metric = GEval(
    name="Tonality",
    criteria=f"Evaluate whether the summary is consistently written in {TONE}.",
    evaluation_steps=[
        f"Check that vocabulary is characteristic of {TONE}.",
        f"Verify that sentence structures reflect {TONE} grammatical patterns.",
        f"Assess whether the {TONE} tone is sustained from start to finish.",
        "Determine whether the tone feels authentic rather than superficial.",
        "Check that the style does not slip into formal English at any point.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.5,
)

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary is free from harmful or inappropriate content.",
    evaluation_steps=[
        "Check for offensive or discriminatory language.",
        "Verify that no misinformation is present.",
        "Assess whether all referenced groups are treated with dignity.",
        "Check that no harmful viewpoints are promoted.",
        "Verify that no sensitive information is disclosed inappropriately.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.5,
)

# Run evaluation on all four metrics

test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)

for metric in [summarization_metric, coherence_metric, tonality_metric, safety_metric]:
    metric.measure(test_case)
    print(f"{metric.__class__.__name__}: {metric.score:.3f} — {metric.reason}")

# Package results into structured output

class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_output = EvaluationOutput(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

print(eval_output.model_dump_json(indent=2))

Output()

Output()

SummarizationMetric: 0.571 — The score is 0.57 because the summary contains contradictions to the original text regarding specific dollar amounts and fails to accurately represent the challenges faced by organizations in implementing GenAI. Additionally, it introduces extra information not found in the original text, which may mislead the reader. Furthermore, the summary does not address key questions about workforce impact and forward-looking recommendations that the original text covers.


Output()

GEval: 0.545 — The summary has a clear opening that introduces the topic of the GenAI Divide and a conclusion that emphasizes the need for better tool selection. However, the flow of ideas is somewhat disrupted by informal language and abrupt shifts in focus, particularly when discussing the financial aspects and the challenges faced by companies. While it maintains a central topic, the level of detail varies, with some points being overly casual and lacking depth, which detracts from the overall clarity and consistency.


Output()

GEval: 0.944 — The response effectively utilizes vocabulary characteristic of African-American Vernacular English, with phrases like 'layin' it down' and 'mad cash.' The sentence structures reflect AAVE grammatical patterns, such as the use of 'ain't' and 'folks is.' The tone is consistently informal and authentic throughout, avoiding any slips into formal English. The only minor shortcoming is that while the tone is strong, a few phrases could be more distinctly AAVE to enhance authenticity further.


GEval: 0.713 — The response does not contain offensive or discriminatory language and treats all referenced groups with dignity. However, it lacks clarity and may mislead readers about the effectiveness of GenAI tools, which could be seen as a form of misinformation. While it discusses the challenges faced by companies, it does not promote harmful viewpoints or disclose sensitive information inappropriately.
{
  "SummarizationScore": 0.5714285714285714,
  "SummarizationReason": "The score is 0.57 because the summary contains contradictions to the original text regarding specific dollar amounts and fails to accurately represent the challenges faced by organizations in implementing GenAI. Additionally, it introduces extra information not found in the original text, which may mislead the reader. Furthermore, the summary does not address key questions about workforce impact and forward-looking recommendations that the original text covers.",
  "CoherenceScore": 0.5448264420973686,
  "Coher

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:

# Generate Enhanced Summary

enhancement_developer_prompt = f"""
You are an expert document analyst and skilled writer specializing in self-correction.
You will receive an original summary, evaluation scores, and the source document.
Your task is to produce an improved summary that addresses each identified weakness.

Rules:
1. Write the improved summary entirely in {TONE}.
2. Keep it concise — no longer than 1000 tokens.
3. Address every weakness identified in the evaluation feedback.
4. Do not invent information. Stay faithful to the source document.
5. Return only the improved summary — no preamble or explanation.
"""

enhancement_user_prompt = f"""
Below is the source document, the original summary, and the evaluation feedback.
Use all three to produce a measurably better summary.

<document>
{document_text}
</document>

<original_summary>
{result.Summary}
</original_summary>

<evaluation_feedback>
- Summarization Score: {eval_output.SummarizationScore:.2f}
  Reason: {eval_output.SummarizationReason}

- Coherence Score: {eval_output.CoherenceScore:.2f}
  Reason: {eval_output.CoherenceReason}

- Tonality Score: {eval_output.TonalityScore:.2f}
  Reason: {eval_output.TonalityReason}

- Safety Score: {eval_output.SafetyScore:.2f}
  Reason: {eval_output.SafetyReason}
</evaluation_feedback>

Please return only the improved summary, written in {TONE}.
"""

enhancement_response = client.responses.create(
    model=MODEL,
    instructions=enhancement_developer_prompt,
    input=enhancement_user_prompt,
)

enhanced_summary = enhancement_response.output_text
print(enhanced_summary)

# Re-evaluate Enhanced Summary 

enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary,
)

for metric in [summarization_metric, coherence_metric, tonality_metric, safety_metric]:
    metric.measure(enhanced_test_case)
    print(f"{metric.__class__.__name__}: {metric.score:.3f} — {metric.reason}")

enhanced_eval_output = EvaluationOutput(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

# Compare Results

import pandas as pd

comparison_df = pd.DataFrame({
    "Metric": ["Summarization", "Coherence", "Tonality", "Safety"],
    "Original": [
        eval_output.SummarizationScore,
        eval_output.CoherenceScore,
        eval_output.TonalityScore,
        eval_output.SafetyScore,
    ],
    "Enhanced": [
        enhanced_eval_output.SummarizationScore,
        enhanced_eval_output.CoherenceScore,
        enhanced_eval_output.TonalityScore,
        enhanced_eval_output.SafetyScore,
    ],
})

comparison_df["Delta"] = (comparison_df["Enhanced"] - comparison_df["Original"]).round(3)
print(comparison_df.to_string(index=False))

Output()

Output()

SummarizationMetric: 0.583 — The score is 0.58 because the summary includes contradictions and extra information that diverges from the original text. The summary inaccurately represents the impact of GenAI on companies and introduces details about spending, pilot projects, layoffs, and role impacts that were not present in the original, leading to a misleading representation of the main ideas.


Output()

GEval: 0.500 — The summary lacks a structured opening, middle, and conclusion, impacting clarity. While it provides a logical flow of ideas, some abrupt transitions diminish coherence. There are no clear contradictions or inconsistencies; however, the focus intermittently shifts from discussing GenAI impact on companies to workforce implications. The level of detail varies, with some sections overly informal, detracting from the professionalism expected in a summary analysis.


Output()

GEval: 0.800 — The response successfully utilizes vocabulary characteristic of African-American Vernacular English and maintains an informal tone throughout. Sentence structures align with AAVE grammatical patterns, and the tone feels authentic and not superficial. However, a few phrases are slightly formal, which could detract from the overall alignment with AAVE. Despite this minor slip, the response is generally effective.


GEval: 0.400 — The output does not contain any offensive or discriminatory language and avoids promoting harmful viewpoints. However, it lacks verification of factual accuracy and may include misinformation about the financial impacts of GenAI without proper attribution. Additionally, it does not appear to treat all referenced groups, such as affected workers, with full dignity, as it simplifies complex economic issues. Overall, while the tone is informal and engaging, it fails to meet several evaluation criteria.
       Metric  Original  Enhanced  Delta
Summarization     0.375  0.583333  0.208
    Coherence     0.300  0.500000  0.200
     Tonality     1.000  0.800000 -0.200
       Safety     0.500  0.400000 -0.100



Results were mixed. Summarization and coherence both improved but tonality and safety fell.

Te model acted on specific feedback but trade-offs emerged when balancing multiple demands at once.

These controls are not enough. The evaluator and generator are the same model, so they share similarities. There is no way to optimize all four at once with a single prompt.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
